# Cleaning and fixing the missing data

#### Cleaning the missing data and calculate the data that need for further processing.
- spreading factor 7
- bandwidth 125 kHz
- coding rate 4/5
- 868 MHzchannel

<table>
  <tr>
    <th colspan="3"> Label and Terrain Penalty (Adjust terrain penalty with your own) </th>
  </tr>
  <tr>
    <th>Label</th>
    <th>Code</th>
    <th>Terrain Penalty</th>
  </tr>
  <tr>
  <!-- first row -->
  <tr>
    <td>Trees</td>
    <td>10</td>
    <td>0.9</td>
  </tr>
  <!-- second row -->
  <tr>
    <td>Shrubland</td>
    <td>20</td>
    <td>0.85</td>
  </tr>
  <!-- third row -->
  <tr>
    <td>Grassland</td>
    <td>30</td>
    <td>0.8</td>
  </tr>
  <!-- fourth row -->
  <tr>
    <td>Cropland</td>
    <td>40</td>
    <td>0.75</td>
  </tr>
  <!-- fifth row -->
  <tr>
    <td>Built-up</td>
    <td>50</td>
    <td>0.6</td>
  </tr>
  <!-- sixth row -->
  <tr>
    <td>Bare/ sparse vegetation</td>
    <td>60</td>
    <td>0.7</td>
  </tr>
  <!-- seventh row -->
  <tr>
    <td>Snow and ice</td>
    <td>70</td>
    <td>0.95</td>
  </tr>
  <!-- eighth row -->
  <tr>
    <td>Permanent water bodies</td>
    <td>80</td>
    <td>0.95</td>
  </tr>
  <!-- ninth row -->
  <tr>
    <td>Herbaceous wetlands</td>
    <td>90</td>
    <td>0.85</td>
  </tr>
  <!-- tenth row -->
  <tr>
    <td>Manggroves</td>
    <td>95</td>
    <td>0.9</td>
  </tr>
  <!-- eleventh row -->
  <tr>
    <td>Moss and lichen</td>
    <td>100</td>
    <td>0.95</td>
  </tr>
</table>


### Preprocessing data

This notebook contains the code to preprocess the raw data for the ABC2026 project.
Code to calculate PDR with SNR threshold and SF with more theoretical sound and accurate way


In [1]:
import pandas as pd
import numpy as np
import ee
import os
from dotenv import load_dotenv
from math import log10, sqrt
from datetime import datetime
from geopy.distance import geodesic # For accurate distance calculation
import warnings
warnings.filterwarnings('ignore')
# Load environment variables from .env file
load_dotenv()

True

#### 1. Initialize Earth Engine

In [2]:
# Ensure you have authenticated using 'earthengine authenticate' in your terminal/command prompt
# before running this script.
ee.Authenticate() # Run this once if you haven't authenticated before

# Try initializing Earth Engine with your project ID
try:
    ee.Initialize(project=os.getenv('GEE_PROJECT_ID'))
    print("Google Earth Engine initialized successfully")
except Exception as e:
    print(f"Google Earth Engine initialization failed: {str(e)}")
    exit()

Google Earth Engine initialized successfully


#### 2. Define Constants (Adjust as Needed)

In [3]:
# Gateway coordinates from the paper
GATEWAYS = {
    'Gateway_24e124fffef07103': {'lat': 28.992878, 'lon': 50.841090},
    'Gateway_24e124fffef06fd1': {'lat': 28.98864,  'lon': 50.83648},
    'Gateway_e45f01fffe376ce6': {'lat': 28.9958761, 'lon': 50.8319432}
}

# Define a plausible destination point (example: a corner of the harbor area)
# Coordinates are approximate and distinct from gateways/data path
DESTINATION_COORDS = {'lat': 28.986, 'lon': 50.840} # Example destination

# ESA WorldCover 10m Land Cover Class Labels
LAND_COVER_LABELS = {
    10: 'Tree',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen'
}

# ESA WorldCover classes: 10 - Tree, 20 - Shrubland, 30 - Grassland, 40 - Cropland,
# 50 - Built-up, 60 - Bare / sparse vegetation, 70 - Snow and ice, 80 - Permanent water bodies, 90 - Herbaceous wetland, 95 - Mangroves, 100 - Moss and lichen
# Define penalties based on obstruction potential (simplified)
PENALTY_MAP = {
    10: 0.9, # Tree - High loss
    20: 0.85, # Shrubland
    30: 0.8, # Grassland
    40: 0.75, # Cropland
    50: 0.6, # Built-up - High loss due to buildings
    60: 0.7, # Bare/sparse
    70: 0.95, # Ice/Snow (reflective, but can block)
    80: 0.95, # Water (reflective, but can block)
    90: 0.85, # Wetland
    95: 0.9, # Mangroves
    100: 0.95 # Moss/lichen
}

# Define frequency and transmit power
FREQUENCY_MHZ = 868
TX_POWER = 14
SPREADING_FACTOR = 7  # Assuming SF7 for LoRa

#### 3. Load and Clean the Raw Data

In [4]:
print("Loading raw data...")
df_raw = pd.read_excel(r'../raw_data/data_1.xlsx', sheet_name='Sheet1')

# Basic inspection
print("\nInitial Data Shape:", df_raw.shape)
print("\nInitial Data Info:")
print(df_raw.info())
print("\nInitial Data Head:")
print(df_raw.head())

# Clean data: Remove rows where Lat/Lon are 0 or missing, or where all RSSI values are 0
def is_valid_location(row):
    # Check if Lat and Lon are not 0 and are not NaN
    return pd.notna(row['Lot']) and pd.notna(row['Lon']) and row['Lot'] != 0 and row['Lon'] != 0

# Apply the location filter
df_clean = df_raw[df_raw.apply(is_valid_location, axis=1)].copy()

print(f"\nData shape after location cleaning: {df_clean.shape}")

# Identify RSSI columns (those starting with 'RSSI(') and not the problematic one)
rssi_cols = [col for col in df_clean.columns if col.startswith('RSSI(') and '(b827eb5555e594df)' not in col]
print(f"\nDetected RSSI columns: {rssi_cols}")

# Focus on the primary gateway mentioned in the PDF and example script: 'RSSI (24e124fffef07103)'
primary_rssi_col = 'RSSI (24e124fffef07103)'
# Find the corresponding SNR column (it's the one immediately after the primary RSSI)
primary_snr_col = df_clean.columns[df_clean.columns.get_loc(primary_rssi_col) + 1]
print(f"Corresponding SNR column for {primary_rssi_col}: {primary_snr_col}")

# Filter rows where the primary RSSI is not 0, -999, or NaN
df_clean = df_clean[(df_clean[primary_rssi_col] != 0) & (df_clean[primary_rssi_col] != -999) & pd.notna(df_clean[primary_rssi_col])]

print(f"\nData shape after filtering for primary RSSI ({primary_rssi_col}): {df_clean.shape}")

Loading raw data...

Initial Data Shape: (1488, 14)

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time                     1488 non-null   object 
 1   Lot                      1488 non-null   float64
 2   Lon                      1488 non-null   float64
 3   DEVID                    1488 non-null   object 
 4   RSSI(b827eb5555e594df)   1488 non-null   int64  
 5   SNR                      1488 non-null   float64
 6   RSSI (24e124fffef07103)  1488 non-null   int64  
 7   SNR                      1488 non-null   float64
 8   RSSI(24e124fffef06f9e)   1488 non-null   int64  
 9   SNR.1                    1488 non-null   float64
 10  RSSI(24e124fffef06fd1)   1488 non-null   int64  
 11  SNR.2                    1488 non-null   float64
 12  RSSI(e45f01fffe376ce6)   1488 non-null   int64  
 13  SNR.3 

#### 4. Prepare Data for Feature Calculation

In [5]:
# Select relevant columns including the corresponding SNR
df_proc = df_clean[['Time', 'Lot', 'Lon', 'DEVID', primary_rssi_col, primary_snr_col]].copy()
df_proc.columns = ['Time', 'latitude', 'longitude', 'DEVID', 'RSSI', 'SNR']  # Rename for clarity

# Convert Time to datetime if needed
df_proc['Time'] = pd.to_datetime(df_proc['Time'])

#### 5. Define Functions for Feature Calculation

In [6]:
def calculate_path_loss(rssi, tx_power_dbm=TX_POWER, freq_mhz=FREQUENCY_MHZ):
    """Calculate path loss based on RSSI, assuming a known transmit power."""
    return tx_power_dbm - rssi

def calculate_elevation(lat, lon):
    """Fetch elevation using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])  # Note: Earth Engine uses [longitude, latitude]
    elevation_dataset = ee.Image('USGS/SRTMGL1_003')  # Or another suitable DEM
    try:
        elevation_value = elevation_dataset.sample(region=point, scale=30).first().get('elevation').getInfo()
        return elevation_value if elevation_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching elevation for {lat}, {lon}: {e}")
        return np.nan

def calculate_land_cover(lat, lon):
    """Fetch land cover code using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    landcover_dataset = ee.ImageCollection('ESA/WorldCover/v100').first()
    try:
        landcover_value = landcover_dataset.sample(region=point, scale=10).first().get('Map').getInfo()
        return landcover_value if landcover_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching land cover for {lat}, {lon}: {e}")
        return np.nan

def land_cover_code_to_label(code):
    """Convert land cover code to a human-readable label."""
    return LAND_COVER_LABELS.get(code, f'Unknown ({code})')

def determine_terrain_penalty(land_cover_code):
    """Map land cover code to a terrain penalty factor."""
    # Default to moderate penalty for unknown land covers
    return PENALTY_MAP.get(land_cover_code, 0.8)

# CORRECTED PDR CALCULATION FUNCTION (Physics-based model for LoRa)
def snr_to_pdr(snr, spreading_factor=7):
    """
    Convert SNR to Packet Delivery Ratio (PDR) for LoRa.
    
    Args:
        snr: Signal-to-Noise Ratio (dB)
        spreading_factor: LoRa spreading factor (7-12)
    
    Returns:
        PDR (0.0 to 1.0)
    """
    # SNR threshold for each spreading factor (from LoRa specification)
    snr_thresholds = {
        7: -7.5,
        8: -10,
        9: -12.5,
        10: -15,
        11: -17.5,
        12: -20
    }
    
    # Get the SNR threshold for the given spreading factor
    snr_threshold = snr_thresholds.get(spreading_factor, -7.5)
    
    # SNR margin above threshold
    margin = snr - snr_threshold
    
    # PDR = 1 - exp(-k * margin) [k = 0.4 from field studies]
    k = 0.4
    pdr = 1 - np.exp(-k * margin)
    
    # Clamp to 0-1 range
    return max(0.0, min(1.0, pdr))

def calculate_distance_to_gateways(lat, lon, gateways_dict):
    """Calculate distances from a point to all gateways and return the minimum (distance_to_start)."""
    distances = {}
    for gw_id, coords in gateways_dict.items():
        dist = geodesic((lat, lon), (coords['lat'], coords['lon'])).meters
        distances[gw_id] = dist
    # Return the distance to the *closest* gateway
    min_distance_to_gateway = min(distances.values()) if distances else np.nan
    closest_gateway_id = [k for k, v in distances.items() if v == min_distance_to_gateway][0] if distances else None
    return min_distance_to_gateway, closest_gateway_id

#### 6. Calculate Features using Earth Engine and Geodesic Distance

In [7]:
print(f"\nCalculating features for {len(df_proc)} data points...")
elevations = []
land_covers = []
path_losses = []
pdrs = []
distances_to_start = []
distances_to_destination = []
closest_gw_ids = []

for idx, row in df_proc.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    rssi = row['RSSI']
    snr = row['SNR']  # Get the corresponding SNR value

    # Calculate distances first
    dist_to_start, closest_gw_id = calculate_distance_to_gateways(lat, lon, GATEWAYS)
    dist_to_dest = geodesic((lat, lon), (DESTINATION_COORDS['lat'], DESTINATION_COORDS['lon'])).meters

    # Fetch other features
    elev = calculate_elevation(lat, lon)
    lc_code = calculate_land_cover(lat, lon)
    pl = calculate_path_loss(rssi)
    pdr = snr_to_pdr(snr, SPREADING_FACTOR)

    distances_to_start.append(dist_to_start)
    distances_to_destination.append(dist_to_dest)
    closest_gw_ids.append(closest_gw_id)
    elevations.append(elev)
    land_covers.append(lc_code)
    path_losses.append(pl)
    pdrs.append(pdr)

    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")

df_proc['distance_to_start'] = distances_to_start
df_proc['distance_to_destination'] = distances_to_destination
df_proc['closest_gateway'] = closest_gw_ids  # Optional: Keep track of which gateway was closest
df_proc['elevation'] = elevations
df_proc['land_cover_code'] = land_covers
df_proc['land_cover_label'] = df_proc['land_cover_code'].apply(land_cover_code_to_label)
df_proc['path_loss'] = path_losses
df_proc['terrain_penalty'] = df_proc['land_cover_code'].apply(determine_terrain_penalty)
df_proc['PDR'] = pdrs

# Placeholder for Latency (if needed for the model, define logic) ---
df_proc['latency_ms'] = 100.0  # Placeholder value


Calculating features for 1268 data points...
Processed 100 rows...
Processed 200 rows...
Processed 300 rows...
Processed 400 rows...
Processed 500 rows...
Processed 700 rows...
Processed 900 rows...
Processed 1000 rows...
Processed 1100 rows...
Processed 1200 rows...
Processed 1300 rows...
Processed 1400 rows...


#### 7. Final Data Preparation

In [8]:
print("\nProcessed Data Shape:", df_proc.shape)
print("\nProcessed Data Info:")
print(df_proc.info())
print("\nProcessed Data Head:")
print(df_proc.head())

# Check for any remaining NaN values in critical columns
print("\nNaN counts in critical columns:")
print(df_proc[['latitude', 'longitude', 'RSSI', 'SNR', 'elevation', 'land_cover_code', 'terrain_penalty', 'path_loss', 'PDR', 'distance_to_start', 'distance_to_destination']].isnull().sum())

# Drop rows where critical features couldn't be calculated (e.g., elevation/land_cover failed)
df_final = df_proc.dropna(subset=['elevation', 'land_cover_code', 'terrain_penalty', 'path_loss', 'PDR', 'distance_to_start', 'distance_to_destination'])
print(f"\nFinal data shape after dropping NaNs: {df_final.shape}")


Processed Data Shape: (1268, 16)

Processed Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1268 entries, 7 to 1487
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Time                     1268 non-null   datetime64[ns]
 1   latitude                 1268 non-null   float64       
 2   longitude                1268 non-null   float64       
 3   DEVID                    1268 non-null   object        
 4   RSSI                     1268 non-null   int64         
 5   SNR                      1268 non-null   float64       
 6   distance_to_start        1268 non-null   float64       
 7   distance_to_destination  1268 non-null   float64       
 8   closest_gateway          1268 non-null   object        
 9   elevation                1268 non-null   int64         
 10  land_cover_code          1268 non-null   int64         
 11  land_cover_label         1268 non-null   obj

#### 8. Save the Processed Dataset

In [10]:
# Reorder columns to match the expected training dataset format
output_cols = ['latitude', 'longitude', 'elevation', 'land_cover_code', 'land_cover_label', 'terrain_penalty',
               'distance_to_start', 'distance_to_destination', 'RSSI', 'SNR', 'PDR', 'latency_ms', 
               'path_loss', 'DEVID', 'Time', 'closest_gateway']
df_output = df_final[output_cols].copy()

# Rename land_cover_code to land_cover for consistency
df_output.rename(columns={'land_cover_code': 'land_cover'}, inplace=True)

print("\nFinal Output Data Shape:", df_output.shape)
print("\nFinal Output Columns:", df_output.columns.tolist())
print("\nFinal Output Head:")
print(df_output.head())

# Save the processed dataset
output_filename = r'../data/processed_data_4.csv'
df_output.to_csv(output_filename, index=False)
print(f"\nProcessed dataset saved as '{output_filename}'")


Final Output Data Shape: (1268, 16)

Final Output Columns: ['latitude', 'longitude', 'elevation', 'land_cover', 'land_cover_label', 'terrain_penalty', 'distance_to_start', 'distance_to_destination', 'RSSI', 'SNR', 'PDR', 'latency_ms', 'path_loss', 'DEVID', 'Time', 'closest_gateway']

Final Output Head:
     latitude  longitude  elevation  land_cover land_cover_label  \
7   28.989470  50.836233          3          50         Built-up   
8   28.989453  50.836219          3          50         Built-up   
9   28.989470  50.836188          3          50         Built-up   
10  28.989456  50.836189          3          50         Built-up   
11  28.989465  50.836227          3          50         Built-up   

    terrain_penalty  distance_to_start  distance_to_destination  RSSI  SNR  \
7               0.6          95.090340               531.672779  -101  8.5   
8               0.6          93.630079               531.256728   -96  6.8   
9               0.6          96.293855              

## Previous preprocessing code -- IGNORE --

#### 1. Code without SNR to calculate PDR

In [ ]:
# Clean data: Remove rows where Lat/Lon are 0 or missing, or where all RSSI values are 0
def is_valid_location(row):
    # Check if Lat and Lon are not 0 and are not NaN
    return pd.notna(row['Lot']) and pd.notna(row['Lon']) and row['Lot'] != 0 and row['Lon'] != 0

# Apply the location filter
df_clean = df_raw[df_raw.apply(is_valid_location, axis=1)].copy()

print(f"\nData shape after location cleaning: {df_clean.shape}")

# Identify RSSI columns (those starting with 'RSSI(') and not the problematic one)
rssi_cols = [col for col in df_clean.columns if col.startswith('RSSI(') and '(b827eb5555e594df)' not in col]
print(f"\nDetected RSSI columns: {rssi_cols}")

# Focus on the primary gateway mentioned in the PDF and example script: 'RSSI (24e124fffef07103)'
primary_rssi_col = 'RSSI (24e124fffef07103)'
# Filter rows where the primary RSSI is not 0, -999, or NaN
df_clean = df_clean[(df_clean[primary_rssi_col] != 0) & (df_clean[primary_rssi_col] != -999) & pd.notna(df_clean[primary_rssi_col])]

print(f"\nData shape after filtering for primary RSSI ({primary_rssi_col}): {df_clean.shape}")

# --- Step 4: Prepare Data for Feature Calculation ---
# Select relevant columns
df_proc = df_clean[['Time', 'Lot', 'Lon', 'DEVID', primary_rssi_col]].copy()
df_proc.columns = ['Time', 'latitude', 'longitude', 'DEVID', 'RSSI'] # Rename for clarity

# Convert Time to datetime if needed
df_proc['Time'] = pd.to_datetime(df_proc['Time'])

# --- Step 5: Define Functions for Feature Calculation ---
def calculate_path_loss(rssi, tx_power_dbm=TX_POWER, freq_mhz=FREQUENCY_MHZ):
    """Calculate path loss based on RSSI, assuming a known transmit power."""
    return tx_power_dbm - rssi

def calculate_elevation(lat, lon):
    """Fetch elevation using Earth Engine."""
    point = ee.Geometry.Point([lon, lat]) # Note: Earth Engine uses [longitude, latitude]
    elevation_dataset = ee.Image('USGS/SRTMGL1_003') # Or another suitable DEM
    try:
        elevation_value = elevation_dataset.sample(region=point, scale=30).first().get('elevation').getInfo()
        return elevation_value if elevation_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching elevation for {lat}, {lon}: {e}")
        return np.nan

def calculate_land_cover(lat, lon):
    """Fetch land cover code using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    landcover_dataset = ee.ImageCollection('ESA/WorldCover/v100').first()
    try:
        landcover_value = landcover_dataset.sample(region=point, scale=10).first().get('Map').getInfo()
        return landcover_value if landcover_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching land cover for {lat}, {lon}: {e}")
        return np.nan

def land_cover_code_to_label(code):
    """Convert land cover code to a human-readable label."""
    return LAND_COVER_LABELS.get(code, f'Unknown ({code})')

def determine_terrain_penalty(land_cover_code):
    """Map land cover code to a terrain penalty factor."""
    
    return PENALTY_MAP.get(land_cover_code, 0.8) # Default to moderate penalty

def calculate_pdr(rssi, snr=None, rssi_threshold=-100, snr_threshold=5):
    """
    Calculate PDR based on RSSI and potentially SNR.
    This is a very simplified model based on thresholds.
    A more robust model would use historical data correlation.
    """
    # Example: PDR increases with better RSSI (less negative)
    # Using a sigmoid-like function based on RSSI for demonstration
    # PDR = 1 / (1 + exp(-(RSSI + 110))) # Adjust offset based on your data's range
    # Or use a simpler threshold-based approach
    if rssi > rssi_threshold: # Adjust threshold based on analysis
        # Further refine based on SNR if available
        if snr is not None and snr > snr_threshold: # Adjust threshold
            return 1.0
        elif snr is not None:
            # Interpolate based on SNR between thresholds
            pdr_snr_factor = min(1.0, max(0.0, (snr - snr_threshold/2) / (snr_threshold/2)))
            return 0.8 + 0.2 * pdr_snr_factor # Base PDR of 0.8 if RSSI is good but SNR is low
        else: # If SNR not available, just use RSSI-based PDR
             # Simple linear mapping from RSSI above threshold to PDR
             # e.g., RSSI -90 -> PDR 1.0, RSSI -100 -> PDR 0.7
             pdr_rssi_factor = min(1.0, max(0.0, (rssi - rssi_threshold) / (-90 - rssi_threshold)))
             return 0.7 + 0.3 * pdr_rssi_factor
    else:
        # If RSSI is below threshold, PDR is low
        return max(0.0, (rssi - (-120)) / (-100 - (-120))) # e.g., RSSI -110 -> PDR 0.5, RSSI -120 -> PDR 0.0

    # Default simple sigmoid
    # return 1 / (1 + np.exp(-(rssi + 110)))

def calculate_distance_to_gateways(lat, lon, gateways_dict):
    """Calculate distances from a point to all gateways and return the minimum (distance_to_start)."""
    distances = {}
    for gw_id, coords in gateways_dict.items():
        dist = geodesic((lat, lon), (coords['lat'], coords['lon'])).meters
        distances[gw_id] = dist
    # Return the distance to the *closest* gateway
    min_distance_to_gateway = min(distances.values()) if distances else np.nan
    closest_gateway_id = [k for k, v in distances.items() if v == min_distance_to_gateway][0] if distances else None
    return min_distance_to_gateway, closest_gateway_id

# --- Step 6: Calculate Features using Earth Engine and Geodesic Distance ---
print(f"\nCalculating features for {len(df_proc)} data points...")
elevations = []
land_covers = []
path_losses = []
pdrs = []
distances_to_start = []
distances_to_destination = []
closest_gw_ids = []

for idx, row in df_proc.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    rssi = row['RSSI']

    # Calculate distances first
    dist_to_start, closest_gw_id = calculate_distance_to_gateways(lat, lon, GATEWAYS)
    dist_to_dest = geodesic((lat, lon), (DESTINATION_COORDS['lat'], DESTINATION_COORDS['lon'])).meters

    # Fetch other features
    elev = calculate_elevation(lat, lon)
    lc_code = calculate_land_cover(lat, lon)
    pl = calculate_path_loss(rssi)
    pdr = calculate_pdr(rssi) # Using simplified model with RSSI only

    distances_to_start.append(dist_to_start)
    distances_to_destination.append(dist_to_dest)
    closest_gw_ids.append(closest_gw_id)
    elevations.append(elev)
    land_covers.append(lc_code)
    path_losses.append(pl)
    pdrs.append(pdr)

    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")

df_proc['distance_to_start'] = distances_to_start
df_proc['distance_to_destination'] = distances_to_destination
df_proc['closest_gateway'] = closest_gw_ids # Optional: Keep track of which gateway was closest
df_proc['elevation'] = elevations
df_proc['land_cover_code'] = land_covers
df_proc['land_cover_label'] = df_proc['land_cover_code'].apply(land_cover_code_to_label)
df_proc['path_loss'] = path_losses
df_proc['terrain_penalty'] = df_proc['land_cover_code'].apply(determine_terrain_penalty)
df_proc['PDR'] = pdrs

# --- Step 7: Add Placeholder for Latency (if needed for the model, define logic) ---
df_proc['latency_ms'] = 100.0 # Placeholder value

# --- Step 8: Final Data Preparation ---
print("\nProcessed Data Shape:", df_proc.shape)
print("\nProcessed Data Info:")
print(df_proc.info())
print("\nProcessed Data Head:")
print(df_proc.head())

# Check for any remaining NaN values in critical columns
print("\nNaN counts in critical columns:")
print(df_proc[['latitude', 'longitude', 'RSSI', 'elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination']].isnull().sum())

# Drop rows where critical features couldn't be calculated (e.g., elevation/land_cover failed)
df_final = df_proc.dropna(subset=['elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination'])
print(f"\nFinal data shape after dropping NaNs: {df_final.shape}")

# --- Step 9: Save the Processed Dataset ---
# Reorder columns to match the expected training dataset format from README
# Include the human-readable land_cover_label
output_cols = ['latitude', 'longitude', 'elevation', 'land_cover_code', 'land_cover_label', 'distance_to_start', 'distance_to_destination', 'RSSI', 'PDR', 'latency_ms', 'path_loss', 'terrain_penalty', 'DEVID', 'Time', 'closest_gateway'] # Added closest_gateway for reference
df_output = df_final[output_cols].copy()

# Rename land_cover_code to land_cover for consistency with README example (keeping label as well)
df_output.rename(columns={'land_cover_code': 'land_cover'}, inplace=True)

print("\nFinal Output Data Shape:", df_output.shape)
print("\nFinal Output Columns:", df_output.columns.tolist())
print("\nFinal Output Head:")
print(df_output.head())

# Save the processed dataset
output_filename = r'../data/processed_data_1.csv' # Or .xlsx if preferred
df_output.to_csv(output_filename, index=False)
# df_output.to_excel('processed_lora_data_for_training.xlsx', index=False) # Uncomment if you prefer Excel

print(f"\nProcessed dataset saved as '{output_filename}'")

Loading raw data...

Initial Data Shape: (1488, 14)

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time                     1488 non-null   object 
 1   Lot                      1488 non-null   float64
 2   Lon                      1488 non-null   float64
 3   DEVID                    1488 non-null   object 
 4   RSSI(b827eb5555e594df)   1488 non-null   int64  
 5   SNR                      1488 non-null   float64
 6   RSSI (24e124fffef07103)  1488 non-null   int64  
 7   SNR                      1488 non-null   float64
 8   RSSI(24e124fffef06f9e)   1488 non-null   int64  
 9   SNR.1                    1488 non-null   float64
 10  RSSI(24e124fffef06fd1)   1488 non-null   int64  
 11  SNR.2                    1488 non-null   float64
 12  RSSI(e45f01fffe376ce6)   1488 non-null   int64  
 13  SNR.3 

#### 2. Code with SNR to calculate PDR

In [ ]:
# Clean data: Remove rows where Lat/Lon are 0 or missing, or where all RSSI values are 0
def is_valid_location(row):
    # Check if Lat and Lon are not 0 and are not NaN
    return pd.notna(row['Lot']) and pd.notna(row['Lon']) and row['Lot'] != 0 and row['Lon'] != 0

# Apply the location filter
df_clean = df_raw[df_raw.apply(is_valid_location, axis=1)].copy()

print(f"\nData shape after location cleaning: {df_clean.shape}")

# Identify RSSI columns (those starting with 'RSSI(') and not the problematic one)
rssi_cols = [col for col in df_clean.columns if col.startswith('RSSI(') and '(b827eb5555e594df)' not in col]
print(f"\nDetected RSSI columns: {rssi_cols}")

# Focus on the primary gateway mentioned in the PDF and example script: 'RSSI (24e124fffef07103)'
primary_rssi_col = 'RSSI (24e124fffef07103)'
# Find the corresponding SNR column (it's the one immediately after the primary RSSI)
primary_snr_col = df_clean.columns[df_clean.columns.get_loc(primary_rssi_col) + 1]
print(f"Corresponding SNR column for {primary_rssi_col}: {primary_snr_col}")

# Filter rows where the primary RSSI is not 0, -999, or NaN
df_clean = df_clean[(df_clean[primary_rssi_col] != 0) & (df_clean[primary_rssi_col] != -999) & pd.notna(df_clean[primary_rssi_col])]

print(f"\nData shape after filtering for primary RSSI ({primary_rssi_col}): {df_clean.shape}")

# --- Step 4: Prepare Data for Feature Calculation ---
# Select relevant columns including the corresponding SNR
df_proc = df_clean[['Time', 'Lot', 'Lon', 'DEVID', primary_rssi_col, primary_snr_col]].copy()
df_proc.columns = ['Time', 'latitude', 'longitude', 'DEVID', 'RSSI', 'SNR'] # Rename for clarity

# Convert Time to datetime if needed
df_proc['Time'] = pd.to_datetime(df_proc['Time'])

# --- Step 5: Define Functions for Feature Calculation ---
def calculate_path_loss(rssi, tx_power_dbm=TX_POWER, freq_mhz=FREQUENCY_MHZ):
    """Calculate path loss based on RSSI, assuming a known transmit power."""
    return tx_power_dbm - rssi

def calculate_elevation(lat, lon):
    """Fetch elevation using Earth Engine."""
    point = ee.Geometry.Point([lon, lat]) # Note: Earth Engine uses [longitude, latitude]
    elevation_dataset = ee.Image('USGS/SRTMGL1_003') # Or another suitable DEM
    try:
        elevation_value = elevation_dataset.sample(region=point, scale=30).first().get('elevation').getInfo()
        return elevation_value if elevation_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching elevation for {lat}, {lon}: {e}")
        return np.nan

def calculate_land_cover(lat, lon):
    """Fetch land cover code using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    landcover_dataset = ee.ImageCollection('ESA/WorldCover/v100').first()
    try:
        landcover_value = landcover_dataset.sample(region=point, scale=10).first().get('Map').getInfo()
        return landcover_value if landcover_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching land cover for {lat}, {lon}: {e}")
        return np.nan

def land_cover_code_to_label(code):
    """Convert land cover code to a human-readable label."""
    return LAND_COVER_LABELS.get(code, f'Unknown ({code})')

def determine_terrain_penalty(land_cover_code):
    """Map land cover code to a terrain penalty factor."""
    
    return PENALTY_MAP.get(land_cover_code, 0.8) # Default to moderate penalty

def calculate_pdr(rssi, snr, rssi_threshold=-100, snr_threshold=5):
    """
    Calculate PDR based on RSSI and SNR.
    This is a simplified model based on thresholds.
    A more robust model would use historical data correlation.
    """
    # Example: PDR increases with better RSSI (less negative) and better SNR (more positive)
    # Using a threshold-based approach with both metrics
    if rssi > rssi_threshold and snr > snr_threshold:
        # Both RSSI and SNR are good
        return 1.0
    elif rssi > rssi_threshold:
        # RSSI is good, but SNR is low
        # Interpolate PDR based on SNR relative to the threshold
        # e.g., if threshold is 5, and SNR is 2, PDR might be 0.7-0.8
        pdr_snr_factor = min(1.0, max(0.0, (snr - snr_threshold/2) / (snr_threshold/2)))
        return 0.7 + 0.3 * pdr_snr_factor # Base PDR of 0.7 if RSSI is good but SNR is low
    elif snr > snr_threshold:
        # SNR is good, but RSSI is low
        # Interpolate PDR based on RSSI relative to the threshold
        # e.g., if threshold is -100, and RSSI is -105, PDR might be 0.2-0.4
        pdr_rssi_factor = min(1.0, max(0.0, (rssi - rssi_threshold) / (-90 - rssi_threshold)))
        return 0.2 + 0.5 * pdr_rssi_factor # Base PDR of 0.2 if SNR is good but RSSI is low
    else:
        # Both RSSI and SNR are below thresholds
        # PDR is low, potentially interpolate based on the worse of the two
        # For simplicity, use a linear decrease based on RSSI (as it's often the primary indicator)
        # e.g., RSSI -110 -> PDR 0.5, RSSI -120 -> PDR 0.0 (assuming SNR is also very low)
        return max(0.0, (rssi - (-120)) / (-100 - (-120))) # e.g., RSSI -110 -> PDR 0.5, RSSI -120 -> PDR 0.0

def calculate_distance_to_gateways(lat, lon, gateways_dict):
    """Calculate distances from a point to all gateways and return the minimum (distance_to_start)."""
    distances = {}
    for gw_id, coords in gateways_dict.items():
        dist = geodesic((lat, lon), (coords['lat'], coords['lon'])).meters
        distances[gw_id] = dist
    # Return the distance to the *closest* gateway
    min_distance_to_gateway = min(distances.values()) if distances else np.nan
    closest_gateway_id = [k for k, v in distances.items() if v == min_distance_to_gateway][0] if distances else None
    return min_distance_to_gateway, closest_gateway_id

# --- Step 6: Calculate Features using Earth Engine and Geodesic Distance ---
print(f"\nCalculating features for {len(df_proc)} data points...")
elevations = []
land_covers = []
path_losses = []
pdrs = []
distances_to_start = []
distances_to_destination = []
closest_gw_ids = []

for idx, row in df_proc.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    rssi = row['RSSI']
    snr = row['SNR'] # Get the corresponding SNR value

    # Calculate distances first
    dist_to_start, closest_gw_id = calculate_distance_to_gateways(lat, lon, GATEWAYS)
    dist_to_dest = geodesic((lat, lon), (DESTINATION_COORDS['lat'], DESTINATION_COORDS['lon'])).meters

    # Fetch other features
    elev = calculate_elevation(lat, lon)
    lc_code = calculate_land_cover(lat, lon)
    pl = calculate_path_loss(rssi)
    pdr = calculate_pdr(rssi, snr) # Pass both RSSI and SNR to the function

    distances_to_start.append(dist_to_start)
    distances_to_destination.append(dist_to_dest)
    closest_gw_ids.append(closest_gw_id)
    elevations.append(elev)
    land_covers.append(lc_code)
    path_losses.append(pl)
    pdrs.append(pdr)

    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")

df_proc['distance_to_start'] = distances_to_start
df_proc['distance_to_destination'] = distances_to_destination
df_proc['closest_gateway'] = closest_gw_ids # Optional: Keep track of which gateway was closest
df_proc['elevation'] = elevations
df_proc['land_cover_code'] = land_covers
df_proc['land_cover_label'] = df_proc['land_cover_code'].apply(land_cover_code_to_label)
df_proc['path_loss'] = path_losses
df_proc['terrain_penalty'] = df_proc['land_cover_code'].apply(determine_terrain_penalty)
df_proc['PDR'] = pdrs

# --- Step 7: Add Placeholder for Latency (if needed for the model, define logic) ---
df_proc['latency_ms'] = 100.0 # Placeholder value

# --- Step 8: Final Data Preparation ---
print("\nProcessed Data Shape:", df_proc.shape)
print("\nProcessed Data Info:")
print(df_proc.info())
print("\nProcessed Data Head:")
print(df_proc.head())

# Check for any remaining NaN values in critical columns
print("\nNaN counts in critical columns:")
print(df_proc[['latitude', 'longitude', 'RSSI', 'SNR', 'elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination']].isnull().sum())

# Drop rows where critical features couldn't be calculated (e.g., elevation/land_cover failed)
df_final = df_proc.dropna(subset=['elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination'])
print(f"\nFinal data shape after dropping NaNs: {df_final.shape}")

# --- Step 9: Save the Processed Dataset ---
# Reorder columns to match the expected training dataset format from README
# Include the human-readable land_cover_label
output_cols = ['latitude', 'longitude', 'elevation', 'land_cover_code', 'land_cover_label', 'distance_to_start', 'distance_to_destination', 'RSSI', 'SNR', 'PDR', 'latency_ms', 'path_loss', 'terrain_penalty', 'DEVID', 'Time', 'closest_gateway'] # Added closest_gateway for reference, included SNR
df_output = df_final[output_cols].copy()

# Rename land_cover_code to land_cover for consistency with README example (keeping label as well)
df_output.rename(columns={'land_cover_code': 'land_cover'}, inplace=True)

print("\nFinal Output Data Shape:", df_output.shape)
print("\nFinal Output Columns:", df_output.columns.tolist())
print("\nFinal Output Head:")
print(df_output.head())

# Save the processed dataset
output_filename = r'../data/processed_data_2.csv' # Or .xlsx if preferred
df_output.to_csv(output_filename, index=False)
# df_output.to_excel('processed_lora_data_for_training.xlsx', index=False) # Uncomment if you prefer Excel

print(f"\nProcessed dataset saved as '{output_filename}'")

Loading raw data...

Initial Data Shape: (1488, 14)

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time                     1488 non-null   object 
 1   Lot                      1488 non-null   float64
 2   Lon                      1488 non-null   float64
 3   DEVID                    1488 non-null   object 
 4   RSSI(b827eb5555e594df)   1488 non-null   int64  
 5   SNR                      1488 non-null   float64
 6   RSSI (24e124fffef07103)  1488 non-null   int64  
 7   SNR                      1488 non-null   float64
 8   RSSI(24e124fffef06f9e)   1488 non-null   int64  
 9   SNR.1                    1488 non-null   float64
 10  RSSI(24e124fffef06fd1)   1488 non-null   int64  
 11  SNR.2                    1488 non-null   float64
 12  RSSI(e45f01fffe376ce6)   1488 non-null   int64  
 13  SNR.3 

#### 3. Code to calculate PDR with SNR threshold and more logical way

In [ ]:
# Clean data: Remove rows where Lat/Lon are 0 or missing, or where all RSSI values are 0
def is_valid_location(row):
    # Check if Lat and Lon are not 0 and are not NaN
    return pd.notna(row['Lot']) and pd.notna(row['Lon']) and row['Lot'] != 0 and row['Lon'] != 0

# Apply the location filter
df_clean = df_raw[df_raw.apply(is_valid_location, axis=1)].copy()

print(f"\nData shape after location cleaning: {df_clean.shape}")

# Identify RSSI columns (those starting with 'RSSI(') and not the problematic one)
rssi_cols = [col for col in df_clean.columns if col.startswith('RSSI(') and '(b827eb5555e594df)' not in col]
print(f"\nDetected RSSI columns: {rssi_cols}")

# Focus on the primary gateway mentioned in the PDF and example script: 'RSSI (24e124fffef07103)'
primary_rssi_col = 'RSSI (24e124fffef07103)'
# Find the corresponding SNR column (it's the one immediately after the primary RSSI)
primary_snr_col = df_clean.columns[df_clean.columns.get_loc(primary_rssi_col) + 1]
print(f"Corresponding SNR column for {primary_rssi_col}: {primary_snr_col}")

# Filter rows where the primary RSSI is not 0, -999, or NaN
# Also filter where the corresponding SNR is not NaN (to ensure we have both values for PDR calculation)
df_clean = df_clean[
    (df_clean[primary_rssi_col] != 0) &
    (df_clean[primary_rssi_col] != -999) &
    pd.notna(df_clean[primary_rssi_col]) &
    pd.notna(df_clean[primary_snr_col]) # Ensure corresponding SNR is also available
]

print(f"\nData shape after filtering for primary RSSI/SNR ({primary_rssi_col}/{primary_snr_col}): {df_clean.shape}")

# --- Step 4: Prepare Data for Feature Calculation ---
# Select relevant columns including the corresponding SNR
df_proc = df_clean[['Time', 'Lot', 'Lon', 'DEVID', primary_rssi_col, primary_snr_col]].copy()
df_proc.columns = ['Time', 'latitude', 'longitude', 'DEVID', 'RSSI', 'SNR'] # Rename for clarity

# Convert Time to datetime if needed
df_proc['Time'] = pd.to_datetime(df_proc['Time'])

# --- Step 5: Define Functions for Feature Calculation ---
def calculate_path_loss(rssi, tx_power_dbm=TX_POWER, freq_mhz=FREQUENCY_MHZ):
    """Calculate path loss based on RSSI, assuming a known transmit power."""
    return tx_power_dbm - rssi

def calculate_elevation(lat, lon):
    """Fetch elevation using Earth Engine."""
    point = ee.Geometry.Point([lon, lat]) # Note: Earth Engine uses [longitude, latitude]
    elevation_dataset = ee.Image('USGS/SRTMGL1_003') # Or another suitable DEM
    try:
        elevation_value = elevation_dataset.sample(region=point, scale=30).first().get('elevation').getInfo()
        return elevation_value if elevation_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching elevation for {lat}, {lon}: {e}")
        return np.nan

def calculate_land_cover(lat, lon):
    """Fetch land cover code using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    landcover_dataset = ee.ImageCollection('ESA/WorldCover/v100').first()
    try:
        landcover_value = landcover_dataset.sample(region=point, scale=10).first().get('Map').getInfo()
        return landcover_value if landcover_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching land cover for {lat}, {lon}: {e}")
        return np.nan

def land_cover_code_to_label(code):
    """Convert land cover code to a human-readable label."""
    return LAND_COVER_LABELS.get(code, f'Unknown ({code})')

def determine_terrain_penalty(land_cover_code):
    """Map land cover code to a terrain penalty factor."""
    
    return PENALTY_MAP.get(land_cover_code, 0.8) # Default to moderate penalty

def calculate_pdr(rssi, snr, critical_snr_threshold=0.0, rssi_threshold=-100):
    """
    Calculate PDR based on RSSI and SNR with a critical SNR check.
    A very low SNR heavily penalizes PDR, even if RSSI is decent.
    """
    # Critical check: If SNR is below the critical threshold, PDR is very low.
    if snr < critical_snr_threshold:
        # Use an exponential decay or a simple low value based on how far below the threshold SNR is.
        # For example, if threshold is 0, SNR -5 gives a lower PDR than SNR -1.
        # A simple model: PDR decreases sharply below critical threshold.
        # e.g., PDR = max(0.0, min(0.1, exp((snr - critical_snr_threshold) / 2)))
        # Or a linear decrease from 0.1 at threshold to 0.0 at very low SNR
        pdr_penalty = max(0.0, min(0.1, (snr - (critical_snr_threshold - 10)) / 10)) # e.g., SNR=0 -> 1.0, SNR=-10 -> 0.0
        return max(0.0, min(0.1, pdr_penalty)) # Ensure PDR is very low but not negative

    # If SNR is acceptable, calculate PDR based on both RSSI and SNR
    # Normalize RSSI and SNR to a 0-1 scale based on thresholds
    # Assume good RSSI is around -80 dBm, poor is around -120 dBm
    # Assume good SNR is around 10 dB, poor is around 0 dB (at threshold)
    rssi_normalized = max(0.0, min(1.0, (rssi - (-120)) / (-80 - (-120))))
    snr_normalized = max(0.0, min(1.0, (snr - critical_snr_threshold) / (10 - critical_snr_threshold)))

    # Combine normalized RSSI and SNR, perhaps weighted or multiplied
    # Multiplication assumes both factors are necessary (AND logic)
    combined_quality = rssi_normalized * snr_normalized
    # Scale this combined quality to a PDR range (e.g., 0.1 to 1.0)
    pdr = 0.1 + 0.9 * combined_quality

    return pdr

def calculate_distance_to_gateways(lat, lon, gateways_dict):
    """Calculate distances from a point to all gateways and return the minimum (distance_to_start)."""
    distances = {}
    for gw_id, coords in gateways_dict.items():
        dist = geodesic((lat, lon), (coords['lat'], coords['lon'])).meters
        distances[gw_id] = dist
    # Return the distance to the *closest* gateway
    min_distance_to_gateway = min(distances.values()) if distances else np.nan
    closest_gateway_id = [k for k, v in distances.items() if v == min_distance_to_gateway][0] if distances else None
    return min_distance_to_gateway, closest_gateway_id

# --- Step 6: Calculate Features using Earth Engine and Geodesic Distance ---
print(f"\nCalculating features for {len(df_proc)} data points...")
elevations = []
land_covers = []
path_losses = []
pdrs = []
distances_to_start = []
distances_to_destination = []
closest_gw_ids = []

for idx, row in df_proc.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    rssi = row['RSSI']
    snr = row['SNR'] # Get the corresponding SNR value

    # Calculate distances first
    dist_to_start, closest_gw_id = calculate_distance_to_gateways(lat, lon, GATEWAYS)
    dist_to_dest = geodesic((lat, lon), (DESTINATION_COORDS['lat'], DESTINATION_COORDS['lon'])).meters

    # Fetch other features
    elev = calculate_elevation(lat, lon)
    lc_code = calculate_land_cover(lat, lon)
    pl = calculate_path_loss(rssi)
    pdr = calculate_pdr(rssi, snr) # Pass both RSSI and SNR to the corrected function

    distances_to_start.append(dist_to_start)
    distances_to_destination.append(dist_to_dest)
    closest_gw_ids.append(closest_gw_id)
    elevations.append(elev)
    land_covers.append(lc_code)
    path_losses.append(pl)
    pdrs.append(pdr)

    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")

df_proc['distance_to_start'] = distances_to_start
df_proc['distance_to_destination'] = distances_to_destination
df_proc['closest_gateway'] = closest_gw_ids # Optional: Keep track of which gateway was closest
df_proc['elevation'] = elevations
df_proc['land_cover_code'] = land_covers
df_proc['land_cover_label'] = df_proc['land_cover_code'].apply(land_cover_code_to_label)
df_proc['path_loss'] = path_losses
df_proc['terrain_penalty'] = df_proc['land_cover_code'].apply(determine_terrain_penalty)
df_proc['PDR'] = pdrs

# --- Step 7: Add Placeholder for Latency (if needed for the model, define logic) ---
df_proc['latency_ms'] = 100.0 # Placeholder value

# --- Step 8: Final Data Preparation ---
print("\nProcessed Data Shape:", df_proc.shape)
print("\nProcessed Data Info:")
print(df_proc.info())
print("\nProcessed Data Head:")
print(df_proc.head())

# Check for any remaining NaN values in critical columns
print("\nNaN counts in critical columns:")
print(df_proc[['latitude', 'longitude', 'RSSI', 'SNR', 'elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination']].isnull().sum())

# Drop rows where critical features couldn't be calculated (e.g., elevation/land_cover failed)
df_final = df_proc.dropna(subset=['elevation', 'land_cover_code', 'path_loss', 'terrain_penalty', 'PDR', 'distance_to_start', 'distance_to_destination'])
print(f"\nFinal data shape after dropping NaNs: {df_final.shape}")

# --- Step 9: Save the Processed Dataset ---
# Reorder columns to match the expected training dataset format from README
# Include the human-readable land_cover_label
output_cols = ['latitude', 'longitude', 'elevation', 'land_cover_code', 'land_cover_label', 'distance_to_start', 'distance_to_destination', 'RSSI', 'SNR', 'PDR', 'latency_ms', 'path_loss', 'terrain_penalty', 'DEVID', 'Time', 'closest_gateway'] # Added closest_gateway for reference, included SNR
df_output = df_final[output_cols].copy()

# Rename land_cover_code to land_cover for consistency with README example (keeping label as well)
df_output.rename(columns={'land_cover_code': 'land_cover'}, inplace=True)

print("\nFinal Output Data Shape:", df_output.shape)
print("\nFinal Output Columns:", df_output.columns.tolist())
print("\nFinal Output Head:")
print(df_output.head())

# Save the processed dataset
output_filename = r'../data/processed_data_3.csv' # Or .xlsx if preferred
df_output.to_csv(output_filename, index=False)
# df_output.to_excel('processed_lora_data_for_training.xlsx', index=False) # Uncomment if you prefer Excel

print(f"\nProcessed dataset saved as '{output_filename}'")

# --- Step 10: Example Analysis of New PDR Calculation ---
print("\n--- Example PDR Calculations (New Method) ---")
test_cases = [
    (-101, 8.5),  # Should be low due to RSSI, but SNR is good
    (-96, 6.8),  # Should be high if both RSSI and SNR are acceptable
    (-97, 9.0),  # Should be high
    (-101, -6.5), # Should be very low due to critical SNR check
    (-80, 5.0),   # Should be medium-high
    (-90, 12.0),  # Should be high
    (-110, 1.0),  # Should be low due to low RSSI and SNR just above critical
    (-70, -2.0),  # Should be low due to critical SNR check
]

for rssi_val, snr_val in test_cases:
    pdr_val = calculate_pdr(rssi_val, snr_val)
    print(f"RSSI={rssi_val}, SNR={snr_val} -> PDR={pdr_val:.3f}")


Loading raw data...

Initial Data Shape: (1488, 14)

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time                     1488 non-null   object 
 1   Lot                      1488 non-null   float64
 2   Lon                      1488 non-null   float64
 3   DEVID                    1488 non-null   object 
 4   RSSI(b827eb5555e594df)   1488 non-null   int64  
 5   SNR                      1488 non-null   float64
 6   RSSI (24e124fffef07103)  1488 non-null   int64  
 7   SNR                      1488 non-null   float64
 8   RSSI(24e124fffef06f9e)   1488 non-null   int64  
 9   SNR.1                    1488 non-null   float64
 10  RSSI(24e124fffef06fd1)   1488 non-null   int64  
 11  SNR.2                    1488 non-null   float64
 12  RSSI(e45f01fffe376ce6)   1488 non-null   int64  
 13  SNR.3 